# 05 — Role classifier on real (external) data

Evaluate TF-IDF + Logistic Regression and MiniLM embeddings + Logistic Regression on the
LiveCareer resume dataset in `data/external/`.

**Dataset** (see top of `docs/EVAL_NOTES.md`): Snehaan Bhawal *Resume Dataset*, licence **CC0 1.0**.

**Pipeline**
1. Load & `light_clean`; drop exact + near-duplicates (TF-IDF cosine > 0.95)
2. Experiment A: original text · Experiment B: role-title words removed (leakage ablation)
3. Stratified **group-aware** split / 5-fold CV; report accuracy, macro/weighted F1, per-class P/R, confusion heatmap
4. Uncertainty: top probability < 0.35 → `uncertain`
5. Top-3 predictions on `tests/eval/resumes/`
6. Save best model to `app/models/job_classifier_v2.joblib` (does **not** overwrite `job_classifier.joblib`)

The heavy lifting lives in `scripts/evaluate_role_classifier.py`; this notebook runs it end-to-end.


In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path().resolve().parent if Path().resolve().name == 'notebooks' else Path().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

%matplotlib inline
from IPython.display import Markdown, display, Image

print("Project root:", PROJECT_ROOT)
print("Dataset:", (PROJECT_ROOT / "data/external/Resume.csv").exists())


## 1–6. Run full evaluation

Cleaning, both experiments, CV, uncertainty rule, eval-resume top-3, model save, and `docs/EVAL_NOTES.md` update.
MiniLM encoding of ~2.5k resumes takes several minutes.


In [ ]:
from scripts.evaluate_role_classifier import main

rc = main()
assert rc == 0, f"evaluation failed with code {rc}"


## Role classifier section (from EVAL_NOTES)


In [ ]:
notes = (PROJECT_ROOT / "docs/EVAL_NOTES.md").read_text(encoding="utf-8")
start = notes.find("## Role classifier")
end = notes.find("\n## ", start + 3)
section = notes[start:] if end < 0 else notes[start:end]
display(Markdown(section))


## Confusion-matrix heatmap


In [ ]:
cm_path = PROJECT_ROOT / "scripts/artifacts/role_classifier_confusion.png"
display(Image(filename=str(cm_path)))
